In [ ]:
# %%
# %% [markdown]
# # Test ONNX Model for YOLOP - Version Corrigée avec Visualisation
# Cette version corrige le preprocessing et ajoute la visualisation debug

# %%
import argparse
import os, sys
import cv2
BASE_DIR = os.getcwd() 
sys.path.append(BASE_DIR)

import pprint
import torch
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
import torchvision.transforms as transforms
import numpy as np

from lib.utils import DataLoaderX
from tensorboardX import SummaryWriter
import lib.dataset as dataset
from lib.config import cfg
from lib.config import update_config
from lib.core.loss import get_loss
from lib.core.function import validate
from lib.core.general import fitness, non_max_suppression
from lib.models import get_net
from lib.utils.utils import create_logger, select_device
import onnxruntime as ort

# %% [markdown]
# ## Configuration

# %%
# ============= CONFIGURATION UTILISATEUR =============
ONNX_MODEL_PATH = 'weights/yolop-1280-1280.onnx'
INPUT_SIZE = 1280  # 640 ou 1280
BATCH_SIZE = 1
CONF_THRES = 0.001
IOU_THRES = 0.6
ENABLE_DEBUG_VIZ = True  # Active la visualisation debug
DEBUG_OUTPUT_DIR = 'runs/debug_viz'
# =====================================================

# %%
def resize_unscale(img, new_shape=(640, 640), color=114):
    """
    Resize avec padding pour garder l'aspect ratio
    Retourne l'image paddée et les informations de transformation
    """
    shape = img.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    canvas = np.zeros((new_shape[0], new_shape[1], 3))
    canvas.fill(color)
    
    # Scale ratio (new / old) new_shape(h,w)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

    # Compute padding
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))  # w,h
    new_unpad_w = new_unpad[0]
    new_unpad_h = new_unpad[1]
    pad_w, pad_h = new_shape[1] - new_unpad_w, new_shape[0] - new_unpad_h  # wh padding

    dw = pad_w // 2  # divide padding into 2 sides
    dh = pad_h // 2

    if shape[::-1] != new_unpad:  # resize
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_AREA)

    canvas[dh:dh + new_unpad_h, dw:dw + new_unpad_w, :] = img

    return canvas, r, dw, dh, new_unpad_w, new_unpad_h


# %%
class ONNXModel(torch.nn.Module):
    """
    Wrapper ONNX avec preprocessing correct et visualisation debug
    """
    def __init__(self, onnx_path, device, input_size=640, debug_viz=False):
        super().__init__()
        self.input_size = input_size
        self.device = device
        self.call_count = 0
        self.debug_viz = debug_viz
        
        if self.debug_viz:
            os.makedirs(DEBUG_OUTPUT_DIR, exist_ok=True)
        
        # Configuration ONNX Runtime
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
        self.session = ort.InferenceSession(onnx_path, providers=providers)
        
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [output.name for output in self.session.get_outputs()]
        
        input_shape = self.session.get_inputs()[0].shape
        print(f"ONNX Input shape: {input_shape}")
        print(f"ONNX Output names: {self.output_names}")
        
        # Attributs requis pour YOLOP validate
        self.names = ['car'] 
        self.nc = 1
        self.gr = 1.0
        self.detector_index = 0
        
    def forward(self, x):
        """
        Forward pass avec preprocessing correct
        x: [B, 3, H, W] tensor déjà normalisé du dataloader
        """
        self.call_count += 1
        batch_size = x.shape[0]
        
        should_print = self.call_count <= 3
        
        if should_print:
            print(f"\n{'='*80}")
            print(f"[Batch {self.call_count}] Input from dataloader: {x.shape}")
        
        # Convertir en numpy et dénormaliser pour appliquer resize_unscale
        x_np = x.cpu().numpy()
        
        # Stockage des paramètres de transformation pour chaque image du batch
        batch_params = []
        processed_imgs = []
        
        for b in range(batch_size):
            # Dénormaliser l'image
            img = x_np[b].transpose(1, 2, 0)  # [3,H,W] -> [H,W,3]
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = (img * 255).astype(np.uint8)
            
            # Appliquer resize_unscale
            canvas, r, dw, dh, new_unpad_w, new_unpad_h = resize_unscale(
                img, (self.input_size, self.input_size)
            )
            
            batch_params.append({
                'r': r, 'dw': dw, 'dh': dh,
                'new_unpad_w': new_unpad_w,
                'new_unpad_h': new_unpad_h
            })
            
            # Renormaliser
            img_norm = canvas.copy().astype(np.float32)
            img_norm /= 255.0
            img_norm[:, :, 0] = (img_norm[:, :, 0] - 0.485) / 0.229
            img_norm[:, :, 1] = (img_norm[:, :, 1] - 0.456) / 0.224
            img_norm[:, :, 2] = (img_norm[:, :, 2] - 0.406) / 0.225
            img_norm = img_norm.transpose(2, 0, 1)  # [H,W,3] -> [3,H,W]
            
            processed_imgs.append(img_norm)
        
        # Stack en batch
        x_processed = np.stack(processed_imgs, axis=0)
        
        if should_print:
            print(f"[Batch {self.call_count}] Shape sent to ONNX: {x_processed.shape}")
        
        # Inférence ONNX
        onnx_outputs = self.session.run(None, {self.input_name: x_processed})
        
        if len(onnx_outputs) != 3:
            raise ValueError(f"Expected 3 outputs, got {len(onnx_outputs)}")
        
        det_out = torch.from_numpy(onnx_outputs[0]).to(self.device)
        da_seg_out = torch.from_numpy(onnx_outputs[1]).to(self.device)
        ll_seg_out = torch.from_numpy(onnx_outputs[2]).to(self.device)
        
        if should_print:
            print(f"\n[ONNX OUTPUTS - Raw]")
            print(f"  - Detection shape: {det_out.shape}")
            print(f"  - DA segmentation shape: {da_seg_out.shape}")
            print(f"  - LL segmentation shape: {ll_seg_out.shape}")
        
        # ==================== POST-TRAITEMENT DES DÉTECTIONS ====================
        # Ajuster les coordonnées selon les paramètres de transformation
        for b in range(batch_size):
            params = batch_params[b]
            r, dw, dh = params['r'], params['dw'], params['dh']
            
            if det_out.dim() == 3 and det_out.shape[-1] >= 4:
                # Les coordonnées sont dans l'espace paddé, il faut les ramener à l'original
                det_out[b, :, 0] = (det_out[b, :, 0] - dw) / r  # x1
                det_out[b, :, 1] = (det_out[b, :, 1] - dh) / r  # y1
                det_out[b, :, 2] = (det_out[b, :, 2] - dw) / r  # x2
                det_out[b, :, 3] = (det_out[b, :, 3] - dh) / r  # y2
        
        if should_print and det_out.dim() == 3:
            coords = det_out[0, :, :4]
            confs = det_out[0, :, 4]
            
            print(f"\n[DETECTION ANALYSIS - After coordinate adjustment]")
            print(f"  - Num predictions: {det_out.shape[1]}")
            print(f"  - Coords range: X=[{coords[:, 0].min():.1f}, {coords[:, 0].max():.1f}], "
                  f"Y=[{coords[:, 1].min():.1f}, {coords[:, 1].max():.1f}]")
            print(f"  - Confidence: min={confs.min():.6f}, max={confs.max():.6f}, mean={confs.mean():.6f}")
            print(f"  - High conf detections (>0.3): {(confs > 0.3).sum().item()}")
        
        # ==================== POST-TRAITEMENT DES SEGMENTATIONS ====================
        # Recadrer les masques de segmentation pour enlever le padding
        da_seg_cropped = []
        ll_seg_cropped = []
        
        for b in range(batch_size):
            params = batch_params[b]
            dw, dh = params['dw'], params['dh']
            new_unpad_w, new_unpad_h = params['new_unpad_w'], params['new_unpad_h']
            
            # Cropper la zone sans padding
            da_crop = da_seg_out[b:b+1, :, dh:dh + new_unpad_h, dw:dw + new_unpad_w]
            ll_crop = ll_seg_out[b:b+1, :, dh:dh + new_unpad_h, dw:dw + new_unpad_w]
            
            # Redimensionner vers la taille du dataset
            target_size = cfg.MODEL.IMAGE_SIZE
            da_resized = torch.nn.functional.interpolate(
                da_crop, size=target_size, mode='bilinear', align_corners=False
            )
            ll_resized = torch.nn.functional.interpolate(
                ll_crop, size=target_size, mode='bilinear', align_corners=False
            )
            
            da_seg_cropped.append(da_resized)
            ll_seg_cropped.append(ll_resized)
        
        da_seg_out = torch.cat(da_seg_cropped, dim=0)
        ll_seg_out = torch.cat(ll_seg_cropped, dim=0)
        
        if should_print:
            print(f"\n[SEGMENTATION POST-PROCESSING]")
            print(f"  - After crop and resize: {da_seg_out.shape}")
        
        # ==================== VISUALISATION DEBUG ====================
        if self.debug_viz and self.call_count <= 5:
            self._save_debug_visualization(
                x_processed[0], det_out[0], da_seg_out[0], ll_seg_out[0],
                batch_params[0], self.call_count
            )
        
        if self.call_count == 3:
            print(f"\n{'='*80}")
            print("[INFO] Logs détaillés désactivés pour les prochains batchs")
            print(f"{'='*80}\n")
        
        # Format de retour pour validate()
        return [[det_out, det_out], da_seg_out, ll_seg_out]
    
    def _save_debug_visualization(self, img_tensor, det_tensor, da_seg, ll_seg, params, batch_num):
        """
        Sauvegarde les visualisations debug comme dans infer_yolop
        """
        try:
            # Dénormaliser l'image
            img = img_tensor.transpose(1, 2, 0)  # [3,H,W] -> [H,W,3]
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = (img * 255).astype(np.uint8)
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            
            # Extraire la zone sans padding
            dw, dh = params['dw'], params['dh']
            new_unpad_w, new_unpad_h = params['new_unpad_w'], params['new_unpad_h']
            img_unpad = img_bgr[dh:dh + new_unpad_h, dw:dw + new_unpad_w]
            
            # Appliquer NMS sur les détections
            det_tensor_batch = det_tensor.unsqueeze(0)  # Add batch dimension
            boxes = non_max_suppression(det_tensor_batch, conf_thres=CONF_THRES, iou_thres=IOU_THRES)[0]
            
            if boxes is not None and len(boxes) > 0:
                boxes = boxes.cpu().numpy().astype(np.float32)
                print(f"[DEBUG VIZ] Detected {boxes.shape[0]} boxes after NMS")
                
                # Dessiner les boîtes
                img_det = img_unpad.copy()
                for i in range(boxes.shape[0]):
                    x1, y1, x2, y2, conf, label = boxes[i]
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                    cv2.rectangle(img_det, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(img_det, f"{conf:.2f}", (x1, y1-5), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                
                cv2.imwrite(f"{DEBUG_OUTPUT_DIR}/batch{batch_num:03d}_detect.jpg", img_det)
            else:
                print(f"[DEBUG VIZ] No boxes detected")
            
            # Traiter les masques de segmentation
            da_seg_np = da_seg.cpu().numpy()
            ll_seg_np = ll_seg.cpu().numpy()
            
            da_seg_mask = np.argmax(da_seg_np, axis=0)  # [H, W]
            ll_seg_mask = np.argmax(ll_seg_np, axis=0)  # [H, W]
            
            # Redimensionner vers la taille non-paddée
            da_seg_mask_resized = cv2.resize(da_seg_mask.astype(np.uint8), 
                                            (new_unpad_w, new_unpad_h), 
                                            interpolation=cv2.INTER_NEAREST)
            ll_seg_mask_resized = cv2.resize(ll_seg_mask.astype(np.uint8), 
                                            (new_unpad_w, new_unpad_h), 
                                            interpolation=cv2.INTER_NEAREST)
            
            # Créer l'image merged
            color_area = np.zeros((new_unpad_h, new_unpad_w, 3), dtype=np.uint8)
            color_area[da_seg_mask_resized == 1] = [0, 255, 0]  # Vert pour driving area
            color_area[ll_seg_mask_resized == 1] = [255, 0, 0]  # Rouge pour lane lines
            
            color_mask = np.mean(color_area, 2)
            img_merge = img_unpad.copy()
            img_merge[color_mask != 0] = img_merge[color_mask != 0] * 0.5 + color_area[color_mask != 0] * 0.5
            img_merge = img_merge.astype(np.uint8)
            
            # Sauvegarder
            cv2.imwrite(f"{DEBUG_OUTPUT_DIR}/batch{batch_num:03d}_merge.jpg", img_merge)
            cv2.imwrite(f"{DEBUG_OUTPUT_DIR}/batch{batch_num:03d}_da.jpg", da_seg_mask_resized * 255)
            cv2.imwrite(f"{DEBUG_OUTPUT_DIR}/batch{batch_num:03d}_ll.jpg", ll_seg_mask_resized * 255)
            
            print(f"[DEBUG VIZ] Saved visualizations to {DEBUG_OUTPUT_DIR}/batch{batch_num:03d}_*.jpg")
            
        except Exception as e:
            print(f"[DEBUG VIZ] Error saving visualization: {e}")


# %%
class DummyCriterion:
    """Criterion factice pour l'inférence"""
    def __call__(self, preds, targets, shapes, model):
        device = preds[0][0].device if isinstance(preds[0], list) else preds[0].device
        zero_loss = torch.tensor(0.0, device=device)
        return zero_loss, [zero_loss.clone(), zero_loss.clone(), zero_loss.clone()]


# %%
def parse_args():
    parser = argparse.ArgumentParser(description='Test ONNX Multitask network')
    parser.add_argument('--modelDir', help='model directory', type=str, default='')
    parser.add_argument('--logDir', help='log directory', type=str, default='runs/')
    parser.add_argument('--weights', nargs='+', type=str, default=[ONNX_MODEL_PATH], help='model.onnx path')
    parser.add_argument('--conf_thres', type=float, default=CONF_THRES, help='object confidence threshold')
    parser.add_argument('--iou_thres', type=float, default=IOU_THRES, help='IOU threshold for NMS')
    
    args = parser.parse_args(args=[])
    return args


# %%
def main():
    # 1. Configuration
    args = parse_args()
    update_config(cfg, args)
    
    cfg.defrost()
    cfg.TEST.BATCH_SIZE_PER_GPU = BATCH_SIZE
    cfg.GPUS = [0] if torch.cuda.is_available() else []
    # La taille cible du dataset (celle vers laquelle on resize les masques de segmentation)
    cfg.MODEL.IMAGE_SIZE = [INPUT_SIZE, INPUT_SIZE]
    cfg.freeze()
    
    print("="*80)
    print(f"Configuration:")
    print(f"  - ONNX Model: {args.weights[0]}")
    print(f"  - ONNX Input Size: {INPUT_SIZE}x{INPUT_SIZE}")
    print(f"  - Dataset Target Size: {cfg.MODEL.IMAGE_SIZE}")
    print(f"  - Batch Size: {BATCH_SIZE}")
    print(f"  - Conf Threshold: {CONF_THRES}")
    print(f"  - IOU Threshold: {IOU_THRES}")
    print(f"  - Debug Visualization: {ENABLE_DEBUG_VIZ}")
    if ENABLE_DEBUG_VIZ:
        print(f"  - Debug Output Dir: {DEBUG_OUTPUT_DIR}")
    print("="*80)
    
    # 2. Logger
    logger, final_output_dir, tb_log_dir = create_logger(cfg, cfg.LOG_DIR, 'test')
    logger.info(pprint.pformat(args))
    logger.info(cfg)
    
    writer_dict = {
        'writer': SummaryWriter(log_dir=tb_log_dir),
        'train_global_steps': 0,
        'valid_global_steps': 0,
    }
    
    # 3. Chargement du Modèle ONNX
    print("\n" + "="*80)
    print("Loading ONNX model...")
    print("="*80)
    
    device = select_device(logger, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU * max(1, len(cfg.GPUS))) \
        if not cfg.DEBUG else select_device(logger, 'cpu')
    
    onnx_path = args.weights[0] if isinstance(args.weights, list) else args.weights
    
    if not os.path.exists(onnx_path):
        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")
    
    logger.info(f"=> Loading ONNX model from '{onnx_path}'")
    model = ONNXModel(onnx_path, device, input_size=INPUT_SIZE, debug_viz=ENABLE_DEBUG_VIZ)
    model.to(device)
    
    print("ONNX model loaded successfully!")
    print("="*80)
    
    # 4. Criterion factice
    criterion = DummyCriterion()
    
    # 5. Chargement des Données
    print("\n" + "="*80)
    print("Loading validation dataset...")
    print("="*80)
    
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
    
    valid_dataset = eval('dataset.' + cfg.DATASET.DATASET)(
        cfg=cfg,
        is_train=False,
        inputsize=cfg.MODEL.IMAGE_SIZE,
        transform=transforms.Compose([
            transforms.ToTensor(),
            normalize,
        ])
    )
    
    valid_loader = DataLoaderX(
        valid_dataset,
        batch_size=cfg.TEST.BATCH_SIZE_PER_GPU * max(1, len(cfg.GPUS)),
        shuffle=False,
        num_workers=cfg.WORKERS,
        pin_memory=False,
        collate_fn=dataset.AutoDriveDataset.collate_fn
    )
    
    print(f"Dataset: {len(valid_dataset)} images, {len(valid_loader)} batches")
    print(f"Images will be processed from 1280x720 with letterbox padding")
    print("="*80)
    
    # 6. Validation
    print("\n" + "="*80)
    print("Starting validation...")
    print("="*80 + "\n")
    
    epoch = 0
    
    da_segment_results, ll_segment_results, detect_results, total_loss, maps, times = validate(
        epoch, cfg, valid_loader, valid_dataset, model, criterion,
        final_output_dir, tb_log_dir, writer_dict,
        logger, device
    )
    
    # 7. Affichage des Résultats
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    
    fi = fitness(np.array(detect_results).reshape(1, -1))
    
    msg = 'Test:    Loss({loss:.3f})\n' \
          'Driving area Segment: Acc({da_seg_acc:.3f})    IOU ({da_seg_iou:.3f})    mIOU({da_seg_miou:.3f})\n' \
          'Lane line Segment: Acc({ll_seg_acc:.3f})    IOU ({ll_seg_iou:.3f})  mIOU({ll_seg_miou:.3f})\n' \
          'Detect: P({p:.3f})  R({r:.3f})  mAP@0.5({map50:.3f})  mAP@0.5:0.95({map:.3f})\n' \
          'Fitness: {fitness:.4f}\n' \
          'Time: inference({t_inf:.4f}s/frame)  nms({t_nms:.4f}s/frame)'.format(
              loss=total_loss, 
              da_seg_acc=da_segment_results[0], da_seg_iou=da_segment_results[1], da_seg_miou=da_segment_results[2],
              ll_seg_acc=ll_segment_results[0], ll_seg_iou=ll_segment_results[1], ll_seg_miou=ll_segment_results[2],
              p=detect_results[0], r=detect_results[1], map50=detect_results[2], map=detect_results[3],
              fitness=fi[0],
              t_inf=times[0], t_nms=times[1])
    
    logger.info(msg)
    print("\n" + msg)
    print("="*80)
    print("Test finished!")
    if ENABLE_DEBUG_VIZ:
        print(f"\nDebug visualizations saved to: {DEBUG_OUTPUT_DIR}/")
    print("="*80)


# %%
if __name__ == '__main__':
    main()

Namespace(modelDir='', logDir='runs/', weights=['weights/yolop-1280-1280.onnx'], conf_thres=0.001, iou_thres=0.6)
Namespace(modelDir='', logDir='runs/', weights=['weights/yolop-1280-1280.onnx'], conf_thres=0.001, iou_thres=0.6)
AUTO_RESUME: False
CUDNN:
  BENCHMARK: True
  DETERMINISTIC: False
  ENABLED: True
DATASET:
  COLOR_RGB: False
  DATAROOT: /workspace/dataset/images
  DATASET: BddDataset
  DATA_FORMAT: jpg
  FLIP: True
  HSV_H: 0.015
  HSV_S: 0.7
  HSV_V: 0.4
  LABELROOT: /workspace/dataset/det_annotations
  LANEROOT: /workspace/dataset/ll_seg_annotations
  MASKROOT: /workspace/dataset/da_seg_annotations
  ORG_IMG_SIZE: [720, 1280]
  ROT_FACTOR: 10
  SCALE_FACTOR: 0.25
  SELECT_DATA: False
  SHEAR: 0.0
  TEST_SET: val
  TRAIN_SET: train
  TRANSLATE: 0.1
DEBUG: False
GPUS: [0]
LOG_DIR: runs/
LOSS:
  BOX_GAIN: 0.05
  CLS_GAIN: 0.5
  CLS_POS_WEIGHT: 1.0
  DA_SEG_GAIN: 0.2
  FL_GAMMA: 0.0
  LL_IOU_GAIN: 0.2
  LL_SEG_GAIN: 0.2
  LOSS_NAME: 
  MULTI_HEAD_LAMBDA: None
  OBJ_GAIN: 1.0


Configuration:
  - ONNX Model: weights/yolop-1280-1280.onnx
  - ONNX Input Size: 1280x1280
  - Dataset Target Size: [1280, 1280]
  - Batch Size: 1
  - Conf Threshold: 0.001
  - IOU Threshold: 0.6
  - Debug Visualization: True
  - Debug Output Dir: runs/debug_viz
=> creating runs/BddDataset/_2026-01-25-20-19

Loading ONNX model...
ONNX Input shape: [1, 3, 1280, 1280]
ONNX Output names: ['det_out', 'drive_area_seg', 'lane_line_seg']
ONNX model loaded successfully!

Loading validation dataset...
building database...


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:02<00:00, 4067.51it/s]

database build finish
Dataset: 10000 images, 10000 batches
Images will be processed from 1280x720 with letterbox padding

Starting validation...




  0%|                                                                                          | 0/10000 [00:00<?, ?it/s]

...

  0%|                                                                                | 4/10000 [00:22<8:53:52,  3.20s/it]

[DEBUG VIZ] Saved visualizations to runs/debug_viz/batch004_*.jpg
[DEBUG VIZ] Detected 300 boxes after NMS


  0%|                                                                                | 6/10000 [00:23<3:59:32,  1.44s/it]

[DEBUG VIZ] Saved visualizations to runs/debug_viz/batch005_*.jpg


 95%|██████████████████████████████████████████████████████████████████████████▊    | 9469/10000 [13:20<00:43, 12.24it/s]